In [ ]:

import unicodedata
from nltk.stem import PorterStemmer

def create_normalized_key(df, column_name, new_column="key"):
    """
    Crea una nueva columna en el DataFrame que contenga una versión normalizada
    de la columna especificada, eliminando tildes y palabras auxiliares.
    """

    # Copia de la columna original
    df[new_column] = df[column_name].astype(str).str.strip().str.lower()

    # 🔹 Quitar tildes
    df[new_column] = df[new_column].apply(
        lambda x: ''.join(
            c for c in unicodedata.normalize('NFKD', x)
            if not unicodedata.combining(c)
        )
    )

    # 🔹 Quitar signos de puntuación
    df[new_column] = df[new_column].str.translate(
        str.maketrans("", "", "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~")
    )

    # 🔹 Tokenizar
    df[new_column] = df[new_column].str.split()

    # 🔹 Lista de palabras auxiliares (stopwords en español comunes)
    stopwords = {
        "de", "del", "la", "las", "el", "los", "en", "y", "o", "para", "por",
        "con", "sin", "una", "uno", "unas", "unos", "al", "a", "su", "sus"
    }

    # 🔹 Eliminar stopwords
    df[new_column] = df[new_column].apply(lambda x: [w for w in x if w not in stopwords])

    # 🔹 Stemming (reducción de palabras)
    stemmer = PorterStemmer()
    df[new_column] = df[new_column].apply(lambda x: [stemmer.stem(word) for word in x])

    # 🔹 Eliminar duplicados y ordenar
    df[new_column] = df[new_column].apply(lambda x: sorted(set(x)))

    # 🔹 Volver a unir las palabras en una sola cadena
    df[new_column] = df[new_column].apply(lambda x: " ".join(x))

    # 🔹 Reemplazar la columna original por la versión normalizada
    df[column_name] = df[new_column]

    # 🔹 Eliminar la columna auxiliar
    df.drop(columns=[new_column], inplace=True, errors="ignore")

    return df
